# 02 · Loading & Compiling the Model

Every forecast needs two steps:

1. **`from_pretrained(...)`** — download / load the weights
2. **`compile(ForecastConfig(...))`** — build the fast batched decode function

You do this **once** per session and reuse the compiled model for all forecasts.

In [ ]:
import torch
import numpy as np
import timesfm

torch.set_float32_matmul_precision("high")

# Downloads ~800 MB of weights the first time, then caches in ~/.cache/huggingface/
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

model.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)
print("Model loaded and compiled.")

## The `ForecastConfig` flags explained

All forecasting behaviour is controlled by `timesfm.ForecastConfig`:

| Parameter | Default | What it does |
| --------- | ------- | ------------ |
| `max_context` | `0` | Longest history window fed to the model (pad/truncate to this). Up to 16,384. |
| `max_horizon` | `0` | Longest forecast the compiled function can produce. |
| `normalize_inputs` | `False` | **Set `True`** — rescales inputs to avoid numerical issues. |
| `use_continuous_quantile_head` | `False` | **Set `True`** for well-calibrated prediction intervals on long horizons. |
| `force_flip_invariance` | `True` | Guarantees `f(-x) = -f(x)`. |
| `infer_is_positive` | `True` | Clamps forecasts ≥ 0 when all inputs are ≥ 0. Set `False` for data that can be negative (temperature, returns). |
| `fix_quantile_crossing` | `False` | **Set `True`** so q10 ≤ q20 ≤ … ≤ q90. |
| `per_core_batch_size` | `1` | Series processed per batch. Tune to your memory. |
| `return_backcast` | `False` | Needed for covariate (XReg) workflows. |

Recompiling is cheap — change a flag and call `model.compile(...)` again.

In [ ]:
# Inspect the model definition (architecture constants)
from timesfm.timesfm_2p5.timesfm_2p5_base import TimesFM_2p5_200M_Definition
d = TimesFM_2p5_200M_Definition()
print("Max context length :", d.context_limit)
print("Input patch length :", d.input_patch_len)
print("Output patch length:", d.output_patch_len)
print("Quantiles          :", d.quantiles)
print("Transformer layers :", d.stacked_transformers.num_layers)
print("Model dims         :", d.stacked_transformers.transformer.model_dims)
print("Attention heads    :", d.stacked_transformers.transformer.num_heads)

### Tip: reuse this loader

In the rest of the series each notebook re-loads the model so it is
self-contained. In your own project, load it **once** and pass `model` around.